In [2]:
!pip install pdfplumber


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 4.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 3.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 3.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pdfplumber]4 [pdfminer.six]


In [1]:
# Core
import os
from pathlib import Path
import re

# Data
import pandas as pd
import numpy as np

# PDF extraction
import pdfplumber

# Progress
from tqdm import tqdm

# Display
pd.set_option("display.max_colwidth", 120)

# Paths
PROJECT_ROOT = Path.cwd().parents[1]
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DRHP_RAW = DATA_RAW / "drhp_raw"

print("Project root:", PROJECT_ROOT)
print("DRHP path exists:", DRHP_RAW.exists())

Project root: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction
DRHP path exists: True


In [2]:
pdf_records = []

for year_dir in sorted(DRHP_RAW.iterdir()):
    if not year_dir.is_dir():
        continue

    year = year_dir.name

    for pdf_file in year_dir.glob("*.pdf"):
        pdf_records.append({
            "year": int(year),
            "pdf_path": pdf_file
        })

pdf_index = pd.DataFrame(pdf_records)

print("Total PDFs found:", len(pdf_index))
pdf_index.head()

Total PDFs found: 314


,year,pdf_path
0,2010,/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/drhp_raw/2010/1287822925470.pdf
1,2010,/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/drhp_raw/2010/1288345787608 15.4...
2,2010,/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/drhp_raw/2010/1288085387201.pdf
3,2010,/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/drhp_raw/2010/1288167592852.pdf
4,2010,/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/drhp_raw/2010/1288346296484.pdf


In [3]:
def extract_pdf_text(pdf_path, max_pages=None):
    """
    Extract full text from a PDF safely.
    """
    text_chunks = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = pdf.pages[:max_pages] if max_pages else pdf.pages

            for page in pages:
                page_text = page.extract_text()
                if page_text:
                    text_chunks.append(page_text)

    except Exception as e:
        return None

    return "\n".join(text_chunks)

In [4]:
texts = []

for row in tqdm(pdf_index.itertuples(index=False), total=len(pdf_index)):
    raw_text = extract_pdf_text(row.pdf_path)

    texts.append({
        "year": row.year,
        "pdf_path": str(row.pdf_path),
        "raw_text": raw_text
    })

drhp_text_df = pd.DataFrame(texts)

drhp_text_df["raw_text"].notna().mean()

  3%|█▎                                      | 10/314 [03:00<1:24:33, 16.69s/it]Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P2' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P6' is an invalid float value
Cannot set gray non-stroke color because /'P7' is an invalid float value
Cannot set gray non-stroke color because /'P3' is an invalid float value
Cannot set gray non-stroke color because /'P4' is an invalid float value
Cannot set gray non-stroke color because /'P5' is an invalid float value
Cannot set gray non-stroke color because /'P

np.float64(0.9968152866242038)

In [5]:
sample_text = drhp_text_df.loc[
    drhp_text_df["raw_text"].notna(), "raw_text"
].iloc[0]

print(sample_text[:3000])

DRAFT RED HERRING PROSPECTUS
Dated June 26, 2010
Please read Section 60 B of Companies Act, 1956
Draft RHP will be updated upon RoC filing
Book Built Issue
RAJPUTANA STAINLESS LIMITED
(Our Company was originally incorporated in India as “Rajputana Steel Castings Private Limited” on April 2, 1991 under the Companies Act, 1956. For details of the change of
name of our Company and Registered Office, see “History and Certain Corporate Matters” on page 119 of this Draft Red Herring Prospectus.)
Registered Office: 213, Madhwas, Halol Kalol Road Tal: Kalol, Dist: Panchmahals, Gujarat – 389 330, India. Tel: 91-2676-326944; Fax: 91-2676-236327
Contact Person: Mr. Amitabh Dubey, Company Secretary and Compliance Officer. Tel: 91-2676-326944; Fax: 91-2676-236327
Email: compliance@rajputanastainless.com; Website: www.rajputanastainless.com
THE PROMOTERS OF THE COMPANY ARE MR. SHANKARLAL D. MEHTA, MR. BABULAL D. MEHTA, AND MR. JAYESH N. PITHWA.
THE ISSUE
PUBLIC ISSUE OF [●] EQUITY SHARES OF RS.10/- 

In [6]:
def extract_company_name(text):
    if not text:
        return None

    patterns = [
        r"([A-Z][A-Z\s&.,]+LIMITED)",
        r"([A-Z][A-Z\s&.,]+PRIVATE LIMITED)"
    ]

    for pat in patterns:
        match = re.search(pat, text[:3000])
        if match:
            return match.group(1).title()

    return None


drhp_text_df["company_name"] = drhp_text_df["raw_text"].apply(extract_company_name)

drhp_text_df["company_name"].notna().mean()

np.float64(0.9299363057324841)

In [7]:
SECTION_PATTERNS = {
    "business": r"(BUSINESS OVERVIEW|OUR BUSINESS|ABOUT THE COMPANY)",
    "risk": r"(RISK FACTORS)",
    "proceeds": r"(OBJECTS OF THE ISSUE|USE OF PROCEEDS)"
}


def extract_sections(text):
    if not text:
        return {}

    sections = {}
    text_upper = text.upper()

    for section, pattern in SECTION_PATTERNS.items():
        match = re.search(pattern, text_upper)
        if match:
            start = match.start()
            sections[section] = text[start:start + 20000]

    return sections

In [8]:
section_rows = []

for row in tqdm(drhp_text_df.itertuples(index=False), total=len(drhp_text_df)):
    sections = extract_sections(row.raw_text)

    for section, text in sections.items():
        section_rows.append({
            "company_name": row.company_name,
            "year": row.year,
            "section": section,
            "text": text,
            "pdf_path": row.pdf_path
        })

section_df = pd.DataFrame(section_rows)

section_df["section"].value_counts()

100%|████████████████████████████████████████| 314/314 [00:02<00:00, 142.01it/s]


section
risk        311
business    302
proceeds    229
Name: count, dtype: int64

In [9]:
section_df = section_df.dropna(subset=["text"])
section_df["text_len"] = section_df["text"].str.len()

section_df = section_df[section_df["text_len"] > 500]

section_df.shape

(842, 6)

In [10]:
OUT_PATH = PROJECT_ROOT / "data" / "processed"
OUT_PATH.mkdir(exist_ok=True, parents=True)

section_df.to_csv(
    OUT_PATH / "drhp_section_text.csv",
    index=False
)

print("Saved:", OUT_PATH / "drhp_section_text.csv")

Saved: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/processed/drhp_section_text.csv


In [11]:
section_df.groupby("section").agg(
    count=("text", "count"),
    avg_len=("text_len", "mean"),
    min_len=("text_len", "min"),
    max_len=("text_len", "max")
)

,count,avg_len,min_len,max_len
section,,,,
business,302,20000.000000,20000,20000
proceeds,229,20000.000000,20000,20000
risk,311,19612.327974,751,20000
